## Lab 1: Creating a simple customer support agent prototype

### Overview

[Amazon Bedrock AgentCore](https://aws.amazon.com/bedrock/agentcore/) helps you deploying and operating AI agents securely at scale - using any framework and model. It provides you with the capability to move from prototype to production faster. 

In this 5-labs tutorial, we will demonstrate the end-to-end journey from prototype to production using a **Customer Support Agent**. For this example we will use [Strands Agents](https://strandsagents.com/latest/), a simple-to-use, code-first framework for building agents and the Anthropic Claude Sonnet 3.7 model from Amazon Bedrock. For your application you can use the framework and model of your choice. It's important to note that the concepts covered here can be applied using other frameworks and models as well.

**Workshop Journey:**
- **Lab 1 (Current)**: Create Agent Prototype - Build a functional customer support agent
- **Lab 2**: Enhance with Memory - Add conversation context and personalization
- **Lab 3**: Scale with Gateway & Identity - Share tools across agents securely
- **Lab 4**: Deploy to Production - Use AgentCore Runtime with observability
- **Lab 5**: Build User Interface - Create a customer-facing application

In this first lab, we'll build a Customer Support Agent prototype that will evolve throughout the workshop into a production-ready system serving multiple customers with persistent memory, shared tools, and full observability. Our agent will have the following local tools available:
- **get_return_policy()** - Get return policy for specific products
- **get_product_info()** - Get product information
- **web_search()** - Search the web for troubleshooting help
- **get_technical_support()** - Search a Bedrock Knowledge Base


### Architecture for Lab 1
<div style="text-align:left">
    <img src="images/architecture_lab1_strands.png" width="75%"/>
</div>

*Simple prototype running locally. In subsequent labs, we'll migrate this to AgentCore services with shared tools, persistent memory, and production-grade observability.*

#### Setup Required Infrastructure

**Important:** Before starting the labs, you need to deploy the required AWS infrastructure (Lambda functions, DynamoDB tables, IAM roles, and Cognito User Pool).

Run the cell below to deploy all required infrastructure. This takes ~5-10 minutes and is needed for Labs 3-5.

In [1]:
# Run prerequisite setup script to deploy CloudFormation resources
# This creates: Lambda functions, DynamoDB tables, IAM roles, and Cognito User Pool
# Required for Labs 3-5
!bash scripts/prereq.sh

🔍 Getting AWS Account ID...
Region: us-east-1
Account ID: 410571135192
🪣 Using S3 bucket: customersupport112-410571135192-us-east-1
{
    "Location": "/customersupport112-410571135192-us-east-1",
    "BucketArn": "arn:aws:s3:::customersupport112-410571135192-us-east-1"
}
Installing dependencies for the Lambda function.
🔧 ARM architecture detected (aarch64), installing Python packages...
  Using cached ddgs-9.11.4-py3-none-any.whl.metadata (14 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached primp-1.1.3-cp310-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (6.7 kB)
  Using cached lxml-6.0.2-cp313-cp313-manylinux_2_26_aarch64.manylinux_2_28_aarch64.whl.metadata (3.6 kB)
Using cached ddgs-9.11.4-py3-none-any.whl (43 kB)
Using cached primp-1.1.3-cp310-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl (3.9 MB)
Using cached click-8.3.1-py3-none-any.whl (108 kB)
Using cached lxml-6.0.2-cp313-cp313-manylinux_2_26_aarch64.manylinux_2_28_aarch64

#### Verify Infrastructure Deployment (Optional)

Run this cell to verify that all required resources were created successfully:

In [2]:
# Verify CloudFormation resources are deployed
import boto3
from scripts.utils import get_ssm_parameter, call_with_retry

try:
    print("Checking deployed resources...\n")
    
    # Check Lambda
    lambda_arn = get_ssm_parameter("/app/customersupport/agentcore/lambda_arn")
    print(f"✅ Lambda Function: {lambda_arn.split(':')[-1]}")
    
    # Check IAM Roles
    gateway_role = get_ssm_parameter("/app/customersupport/agentcore/gateway_iam_role")
    print(f"✅ Gateway IAM Role: {gateway_role.split('/')[-1]}")
    
    # Check Cognito
    cognito_url = get_ssm_parameter("/app/customersupport/agentcore/cognito_discovery_url")
    print(f"✅ Cognito User Pool: Configured")
    
    # Check DynamoDB Tables
    warranty_table = get_ssm_parameter("/app/customersupport/dynamodb/warranty_table_name")
    customer_table = get_ssm_parameter("/app/customersupport/dynamodb/customer_profile_table_name")
    print(f"✅ DynamoDB Tables: {warranty_table}, {customer_table}")
    
    print("\n🎉 All infrastructure deployed successfully!")
    
except Exception as e:
    print(f"❌ Error: {str(e)}")
    print("\nPlease ensure the prereq.sh script completed successfully.")

Checking deployed resources...

✅ Lambda Function: CustomerSupportStackInfra-CustomerSupportLambda-xaoXFAgXEPTN
✅ Gateway IAM Role: CustomerSupportStackInfra-GatewayAgentCoreRole-sMBN0rl7e2YW
✅ Cognito User Pool: Configured
✅ DynamoDB Tables: CustomerSupportStackInfra-WarrantyTable-B485WPTJJ1N, CustomerSupportStackInfra-CustomerProfileTable-1QPYW00CI4KRS

🎉 All infrastructure deployed successfully!


### Step 1: Import Libraries

We can now import the required libraries and initialize our boto3 session

In [3]:
# Import libraries
import boto3
from boto3.session import Session

from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

from strands.tools import tool

In [4]:
# Get boto session
boto_session = Session()
region = boto_session.region_name

### Step 2: Implementing custom tools

Next, we will implement the 3 tools which will be provided to the Customer Support Agent.

Defining tools in Strands Agent is extremely simple, just add a `@tool` decorator to your function, and provide a description of the tool in the function's docstring. Strands Agents will use the function documentation, typing and arguments to provide context on this tool to your agent. 


#### Tool 1: Get Return Policy

**Purpose:** This tool helps customers understand return policies for different product categories. It provides detailed information about return windows, conditions, processes, and refund timelines so customers know exactly what to expect when returning items.

In [5]:
@tool
def get_return_policy(product_category: str) -> str:
    """
    Get return policy information for a specific product category.

    Args:
        product_category: Electronics category (e.g., 'smartphones', 'laptops', 'accessories')

    Returns:
        Formatted return policy details including timeframes and conditions
    """
    # Mock return policy database - in real implementation, this would query policy database
    return_policies = {
        "smartphones": {
            "window": "30 days",
            "condition": "Original packaging, no physical damage, factory reset required",
            "process": "Online RMA portal or technical support",
            "refund_time": "5-7 business days after inspection",
            "shipping": "Free return shipping, prepaid label provided",
            "warranty": "1-year manufacturer warranty included"
        },
         "laptops": {
            "window": "30 days", 
            "condition": "Original packaging, all accessories, no software modifications",
            "process": "Technical support verification required before return",
            "refund_time": "7-10 business days after inspection",
            "shipping": "Free return shipping with original packaging",
            "warranty": "1-year manufacturer warranty, extended options available"
        },
        "accessories": {
            "window": "30 days",
            "condition": "Unopened packaging preferred, all components included",
            "process": "Online return portal",
            "refund_time": "3-5 business days after receipt",
            "shipping": "Customer pays return shipping under $50",
            "warranty": "90-day manufacturer warranty"
        }
    }

    # Default policy for unlisted categories
    default_policy = {
        "window": "30 days",
        "condition": "Original condition with all included components",
        "process": "Contact technical support",
        "refund_time": "5-7 business days after inspection", 
        "shipping": "Return shipping policies vary",
        "warranty": "Standard manufacturer warranty applies"
    }

    policy = return_policies.get(product_category.lower(), default_policy)
    return f"Return Policy - {product_category.title()}:\n\n" \
           f"• Return window: {policy['window']} from delivery\n" \
           f"• Condition: {policy['condition']}\n" \
           f"• Process: {policy['process']}\n" \
           f"• Refund timeline: {policy['refund_time']}\n" \
           f"• Shipping: {policy['shipping']}\n" \
           f"• Warranty: {policy['warranty']}"
print("✅ Return policy tool ready")

✅ Return policy tool ready


#### Tool 2: Get Product Information

**Purpose:** This tool provides customers with comprehensive product details including warranties, available models, key features, shipping policies, and return information. It helps customers make informed purchasing decisions and understand what they're buying.

In [6]:
@tool
def get_product_info(product_type: str) -> str:
    """
    Get detailed technical specifications and information for electronics products.

    Args:
        product_type: Electronics product type (e.g., 'laptops', 'smartphones', 'headphones', 'monitors')
    Returns:
        Formatted product information including warranty, features, and policies
    """
    # Mock product catalog - in real implementation, this would query a product database
    products = {
        "laptops": {
            "warranty": "1-year manufacturer warranty + optional extended coverage",
            "specs": "Intel/AMD processors, 8-32GB RAM, SSD storage, various display sizes",
            "features": "Backlit keyboards, USB-C/Thunderbolt, Wi-Fi 6, Bluetooth 5.0",
            "compatibility": "Windows 11, macOS, Linux support varies by model",
            "support": "Technical support and driver updates included"
        },
        "smartphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "5G/4G connectivity, 128GB-1TB storage, multiple camera systems",
            "features": "Wireless charging, water resistance, biometric security",
            "compatibility": "iOS/Android, carrier unlocked options available",
            "support": "Software updates and technical support included"
        },
        "headphones": {
            "warranty": "1-year manufacturer warranty",
            "specs": "Wired/wireless options, noise cancellation, 20Hz-20kHz frequency",
            "features": "Active noise cancellation, touch controls, voice assistant",
            "compatibility": "Bluetooth 5.0+, 3.5mm jack, USB-C charging",
            "support": "Firmware updates via companion app"
        },
        "monitors": {
            "warranty": "3-year manufacturer warranty",
            "specs": "4K/1440p/1080p resolutions, IPS/OLED panels, various sizes",
            "features": "HDR support, high refresh rates, adjustable stands",
            "compatibility": "HDMI, DisplayPort, USB-C inputs",
            "support": "Color calibration and technical support"
        }
    }
    product = products.get(product_type.lower())
    if not product:
        return f"Technical specifications for {product_type} not available. Please contact our technical support team for detailed product information and compatibility requirements."

    return f"Technical Information - {product_type.title()}:\n\n" \
           f"• Warranty: {product['warranty']}\n" \
           f"• Specifications: {product['specs']}\n" \
           f"• Key Features: {product['features']}\n" \
           f"• Compatibility: {product['compatibility']}\n" \
           f"• Support: {product['support']}"

print("✅ get_product_info tool ready")

✅ get_product_info tool ready


#### Tool 3: Web-search

**Purpose:** This tool allows customers to get troubleshooting support or suggestions on product recommendations etc.

In [7]:
@tool
def web_search(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"

print("✅ Web search tool ready")

✅ Web search tool ready


#### Customer Support Agent - Knowledge Base Integration Steps

##### Download product technical_support files from S3

In [8]:
import os

def download_files():
    # Get account and region
    account_id = boto3.client('sts').get_caller_identity()['Account']
    region = boto3.Session().region_name
    bucket_name = f"{account_id}-{region}-kb-data-bucket"
    
    # Create local folder
    os.makedirs("knowledge_base_data", exist_ok=True)
    
    # Download all files
    s3 = boto3.client('s3')
    objects = s3.list_objects_v2(Bucket=bucket_name)
    
    for obj in objects['Contents']:
        file_name = obj['Key']
        s3.download_file(bucket_name, file_name, f"knowledge_base_data/{file_name}")
        print(f"Downloaded: {file_name}")
    
    print(f"All files saved to: knowledge_base_data/")

# Run it
download_files()

Downloaded: laptop-maintenance-guide.txt
Downloaded: monitor-calibration-guide.txt
Downloaded: smartphone-setup-guide.txt
Downloaded: troubleshooting-guide.txt
Downloaded: warranty-service-guide.txt
Downloaded: wireless-connectivity-guide.txt
All files saved to: knowledge_base_data/


#### Knowledge Base Sync Job

##### Sync the knowledge base with product technical_support files from S3 which can be integrated with the agent

In [9]:
import boto3
import time

# Get parameters
ssm = boto3.client('ssm')
bedrock = boto3.client('bedrock-agent')
s3 = boto3.client('s3')

account_id = boto3.client('sts').get_caller_identity()['Account']
region = boto3.Session().region_name

kb_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")['Parameter']['Value']
ds_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/data-source-id")['Parameter']['Value']

# Get file names from S3 bucket
bucket_name = f"{account_id}-{region}-kb-data-bucket"
s3_objects = s3.list_objects_v2(Bucket=bucket_name)
file_names = [obj['Key'] for obj in s3_objects.get('Contents', [])]

# Start sync job
response = bedrock.start_ingestion_job(
    knowledgeBaseId=kb_id,
    dataSourceId=ds_id,
    description="Quick sync"
)

job_id = response['ingestionJob']['ingestionJobId']
print("Bedrock knowledge base sync job started, ingesting the data files from s3")

# Monitor until complete
while True:
    job = bedrock.get_ingestion_job(
        knowledgeBaseId=kb_id,
        dataSourceId=ds_id,
        ingestionJobId=job_id
    )['ingestionJob']
    
    status = job['status']
    
    if status in ['COMPLETE', 'FAILED']:
        break
        
    time.sleep(10)

# Print final result
if status == 'COMPLETE':
    file_count = job.get('statistics', {}).get('numberOfDocumentsScanned', 0)
    files_list = ', '.join(file_names)
    print(f"Bedrock knowledge base sync job completed Successfully, ingested {file_count} files")
    print(f"Files ingested: {files_list}")
else:
    print(f"Bedrock knowledge base sync job failed with status: {status}")

Bedrock knowledge base sync job started, ingesting the data files from s3
Bedrock knowledge base sync job completed Successfully, ingested 6 files
Files ingested: laptop-maintenance-guide.txt, monitor-calibration-guide.txt, smartphone-setup-guide.txt, troubleshooting-guide.txt, warranty-service-guide.txt, wireless-connectivity-guide.txt


#### Tool 4: Get Technical Support

**Purpose:**  This tool provides customers with comprehensive technical support and troubleshooting assistance by accessing our knowledge base of electronics documentation. It includes detailed setup guides, maintenance instructions, troubleshooting steps, connectivity solutions, and warranty service information. This tool helps customers resolve technical issues, properly configure their devices, and understand maintenance requirements for optimal product performance.

In [10]:
import os
from strands.tools.mcp.mcp_client import MCPClient
from mcp.client.stdio import stdio_client
from mcp.client.stdio import StdioServerParameters
from strands.models import BedrockModel
from strands import Agent
from strands_tools import retrieve

@tool
def get_technical_support(issue_description: str) -> str:
	try:
		# Get KB ID from parameter store
		ssm = boto3.client('ssm')
		account_id = boto3.client('sts').get_caller_identity()['Account']
		region = boto3.Session().region_name

		kb_id = ssm.get_parameter(Name=f"/{account_id}-{region}/kb/knowledge-base-id")['Parameter']['Value']
		print(f"Successfully retrieved KB ID: {kb_id}")

		# Use strands retrieve tool
		tool_use = {
			"toolUseId": "tech_support_query",
			"input": {
				"text": issue_description,
				"knowledgeBaseId": kb_id,
				"region": region,
				"numberOfResults": 3,
				"score": 0.4
			}
		}

		result = retrieve.retrieve(tool_use)

		if result["status"] == "success":
			return result["content"][0]["text"]
		else:
			return f"Unable to access technical support documentation. Error: {result['content'][0]['text']}"

	except Exception as e:
		print(f"Detailed error in get_technical_support: {str(e)}")
		return f"Unable to access technical support documentation. Error: {str(e)}"

print("✅ Technical support tool ready")

✅ Technical support tool ready


### Step 4: Create and Configure the Customer Support Agent

Next, we will create the Customer Support Agent providing a model, the list of tools implemented in the previous step, and with a system prompt.

In [11]:
from strands import Agent
from strands.models import BedrockModel

SYSTEM_PROMPT = """You are a helpful and professional customer support assistant for an electronics e-commerce company.
Your role is to:
- Provide accurate information using the tools available to you
- Support the customer with technical information and product specifications, and maintenance questions
- Be friendly, patient, and understanding with customers
- Always offer additional help after answering questions
- If you can't help with something, direct customers to the appropriate contact

You have access to the following tools:
1. get_return_policy() - For warranty and return policy questions
2. get_product_info() - To get information about a specific product
3. web_search() - To access current technical documentation, or for updated information. 
4. get_technical_support() - For troubleshooting issues, setup guides, maintenance tips, and detailed technical assistance
For any technical problems, setup questions, or maintenance concerns, always use the get_technical_support() tool as it contains our comprehensive technical documentation and step-by-step guides.

Always use the appropriate tool to get accurate, up-to-date information rather than making assumptions about electronic products or specifications."""

# Initialize the Bedrock model (Anthropic Claude 4 Sonnet)
model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
    temperature=0.3,
    region_name=region
)

# Create the customer support agent with all tools
agent = Agent(
    model=model,
    tools=[
        get_product_info,  # Tool 1: Simple product information lookup
        get_return_policy,  # Tool 2: Simple return policy lookup
        web_search, # Tool 3: Access the web for updated information
        get_technical_support   # Tool 4: Technical support & troubleshooting
    ],
    system_prompt=SYSTEM_PROMPT,
)

print("Customer Support Agent created successfully!")

Customer Support Agent created successfully!


### Step 5: Test the Customer Support Agent

Let's test our agent with sample queries to ensure all tools work correctly.

#### Test Return check

In [12]:
call_with_retry(lambda: agent("What's the return policy for my thinkpad X1 Carbon?"))

I'll help you find the return policy information for your ThinkPad X1 Carbon laptop.
Tool #1: get_return_policy
Here's the return policy for your ThinkPad X1 Carbon laptop:

**Return Window:** 30 days from delivery date

**Return Conditions:**
- Must be in original packaging
- All accessories must be included
- No software modifications allowed

**Return Process:**
- Technical support verification is required before initiating the return
- Free return shipping when using original packaging

**Refund Timeline:** 7-10 business days after our inspection is complete

**Warranty Information:** Your laptop comes with a 1-year manufacturer warranty, with extended warranty options available for purchase.

Is there anything specific about the return process you'd like me to clarify? I'm also here to help if you're experiencing any technical issues with your ThinkPad that might be resolved without needing to return it.

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here's the return policy for your ThinkPad X1 Carbon laptop:\n\n**Return Window:** 30 days from delivery date\n\n**Return Conditions:**\n- Must be in original packaging\n- All accessories must be included\n- No software modifications allowed\n\n**Return Process:**\n- Technical support verification is required before initiating the return\n- Free return shipping when using original packaging\n\n**Refund Timeline:** 7-10 business days after our inspection is complete\n\n**Warranty Information:** Your laptop comes with a 1-year manufacturer warranty, with extended warranty options available for purchase.\n\nIs there anything specific about the return process you'd like me to clarify? I'm also here to help if you're experiencing any technical issues with your ThinkPad that might be resolved without needing to return it."}]}, metrics=EventLoopMetrics(cycle_count=2, tool_metrics={'get_return_policy': Tool

In [13]:
call_with_retry(lambda: agent("My laptop won't turn on, what should I check?"))

I'll help you troubleshoot your laptop that won't turn on. Let me get you the proper technical support guidance for this issue.
Tool #2: get_technical_support
Successfully retrieved KB ID: D39IAEEYYQ
Based on our technical support documentation, here's a systematic approach to troubleshoot your laptop that won't turn on:

## **Step-by-Step Troubleshooting:**

### **1. Power Connection Checks:**
- **Verify power cable connections** - Make sure the power adapter is firmly connected to both your laptop and the wall outlet
- **Test the power outlet** - Try plugging another device into the same outlet to confirm it's working
- **Try a different power cable** if you have one available
- **Check for physical damage** to the power adapter, cable, or charging port

### **2. Basic Power Reset:**
- Remove the battery (if removable) and unplug the power adapter
- Hold the power button for 15-30 seconds to discharge any residual power
- Reconnect everything and try turning it on

### **3. Check for

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Based on our technical support documentation, here's a systematic approach to troubleshoot your laptop that won't turn on:\n\n## **Step-by-Step Troubleshooting:**\n\n### **1. Power Connection Checks:**\n- **Verify power cable connections** - Make sure the power adapter is firmly connected to both your laptop and the wall outlet\n- **Test the power outlet** - Try plugging another device into the same outlet to confirm it's working\n- **Try a different power cable** if you have one available\n- **Check for physical damage** to the power adapter, cable, or charging port\n\n### **2. Basic Power Reset:**\n- Remove the battery (if removable) and unplug the power adapter\n- Hold the power button for 15-30 seconds to discharge any residual power\n- Reconnect everything and try turning it on\n\n### **3. Check for Signs of Life:**\n- Look for any LED indicators when plugged in\n- Listen for fan noise or hard 

#### Test Troubleshooting

In [15]:
call_with_retry(lambda: agent("I bought an iphone 14 last month. I don't like it because it heats up. How do I solve it?"))

I'll help you troubleshoot the heating issue with your iPhone 14. Let me get you the proper technical guidance for this problem.
Tool #3: get_technical_support
Successfully retrieved KB ID: D39IAEEYYQ
I can see you're experiencing overheating with your iPhone 14. Let me get more specific technical guidance for smartphone heating issues.
Tool #4: get_technical_support
Successfully retrieved KB ID: D39IAEEYYQ
Based on our technical support documentation, here are the steps to address your iPhone 14 heating issue:

## **Immediate Solutions for Overheating:**

### **1. Check for Overheating Causes:**
- **Close unnecessary applications** running in the background
- **Check available storage space** - low storage can cause overheating
- **Run system updates** - ensure your iOS is up to date
- **Clear cache and temporary files**

### **2. Environmental Factors:**
- **Avoid direct sunlight** or hot environments
- **Remove the phone case** temporarily to allow better heat dissipation
- **Don't 

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Based on our technical support documentation, here are the steps to address your iPhone 14 heating issue:\n\n## **Immediate Solutions for Overheating:**\n\n### **1. Check for Overheating Causes:**\n- **Close unnecessary applications** running in the background\n- **Check available storage space** - low storage can cause overheating\n- **Run system updates** - ensure your iOS is up to date\n- **Clear cache and temporary files**\n\n### **2. Environmental Factors:**\n- **Avoid direct sunlight** or hot environments\n- **Remove the phone case** temporarily to allow better heat dissipation\n- **Don't use the phone while charging** if possible\n- **Keep the phone on hard, flat surfaces** for proper ventilation\n\n### **3. Settings Adjustments:**\n- **Reduce screen brightness**\n- **Turn off background app refresh** for apps you don't need\n- **Disable location services** for non-essential apps\n- **Enable 

## 🎉 Lab 1 Complete!

You've successfully created a functional Customer Support Agent prototype! Here's what you accomplished:

- Built an agent with 3 custom tools (return policy, product info, web search)  
- Tested multi-tool interactions and web search capabilities  
- Established the foundation for our production journey  

### Current Limitations (We'll fix these!)
- **Single user conversation memory** - local conversation session, multiple customers need multiple sessions.
- **Conversation history limited to session** - no long term memory or cross session information is available in the conversation.
- **Tools reusability** - tools aren't reusable across different agents  
- **Running locally only** - not scalable
- **Identity** - No user and/or agent identity or access control
- **Observability** - Limited observability into agent behavior
- **Existing APIs** - No access to existing enterprise APIs for customer data

##### Next Up [Lab 2: Personalize our agent by adding memory →](lab-02-agentcore-memory.ipynb)
